# E0.3d.1 — Response of the **actual Bollinger bands** to BTC price impulses

Extends E0.3d without modifying its containment/width/crossing benchmarks. The **(20, 2.0)** configuration remains frozen. Full-year 2025 study, with 4H/1H/15m/5m and identical ATR-derived reference impulses for every parameter pair **within** each timeframe. Different timeframes have different reference populations.

**Measurement conventions:** Reference onset is retrospective; these metrics are offline diagnostics, **not** causal entry signals. Bands use only data through their own completed candle. Band response is compared with the last band known before reference onset. No parameter is selected automatically, and faster band response alone is not assumed to be better breakout detection.

In [2]:
import sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    pass

import ss_bands as sb
from walk_states import bollinger_from_5m
from e03c_analysis import price_impulse_references
from e03d_surface import surface, plot_heatmap, WINDOWS, SIGMAS
from e03d_band_response import impulse_response_details, summarize_impulse_response

START, END, SCORE_START = '2025-01-01', '2026-01-01', '2025-01-03'
TF = {'4H':'4h','1H':'1h','15m':'15min','5m':'5min'}

DATA_SOURCE = "binance"      # "binance" (your cache/API) or "synthetic" (code check only)

START = pd.Timestamp("2024-12-30", tz="UTC")   # warm-up before 2025
END = pd.Timestamp("2026-01-02", tz="UTC")     # exclusive
EVAL_START = pd.Timestamp("2025-01-01", tz="UTC")
TRAIN_END = pd.Timestamp("2025-07-01", tz="UTC")   # train = H1, test = H2
EVAL_END = pd.Timestamp("2026-01-01", tz="UTC")
NOV_START, NOV_END = "2025-11-13", "2025-11-23"

TARGET_CONTAINMENT = 0.95
SHOCK_SIGMA = 5.0          # shock = |5m log return| > 5 x EWMA std known at t-1
HORIZONS = [6, 12, 24]     # forward-return horizons in bars (30m, 1h, 2h)
COST_BPS = 20.0            # round trip: 0.10% taker each side; use 15 with BNB discount

REPO = Path("/media/ata/Data/personal/binbot")
DATA_DIR = REPO / "research" / "sentiment_price" / "data"
CACHE_DIR = DATA_DIR / "btcusdt_5m_batches"
STEP_MS = 5 * 60 * 1000
START_MS = int(START.timestamp() * 1000)
END_MS = int(END.timestamp() * 1000)


In [3]:
def load_binance_5m():
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    all_candles = []
    cursor = START_MS
    with requests.Session() as session:
        while cursor < END_MS:
            batch_end = min(cursor + 1000 * STEP_MS, END_MS)
            cache_file = CACHE_DIR / f"{cursor}_{batch_end}.json"
            if cache_file.exists():
                batch = json.loads(cache_file.read_text())
            else:
                response = session.get(
                    "https://api.binance.com/api/v3/klines",
                    params={"symbol": "BTCUSDT", "interval": "5m", "timeZone": "0",
                            "startTime": cursor, "endTime": batch_end - 1, "limit": 1000},
                    timeout=30,
                )
                response.raise_for_status()
                batch = response.json()
            if not isinstance(batch, list) or not batch:
                raise ValueError(f"Unexpected response at {cursor}: {batch}")
            expected_opens = list(range(cursor, batch_end, STEP_MS))
            actual_opens = [int(row[0]) for row in batch]
            if actual_opens != expected_opens:
                raise ValueError(f"Missing, duplicate or unexpected candles in batch {cursor}")
            if not cache_file.exists():
                temporary = cache_file.with_suffix(".tmp")
                temporary.write_text(json.dumps(batch))
                temporary.replace(cache_file)
                time.sleep(0.2)
            all_candles.extend(batch)
            cursor = batch_end
            print(f"\rLoaded {len(all_candles):,} candles", end="", flush=True)
    print()
    columns = ["open_time", "open", "high", "low", "close", "volume", "close_time",
               "quote_volume", "trade_count", "taker_buy_volume", "taker_buy_quote_volume", "ignore"]
    bars = pd.DataFrame(all_candles, columns=columns)
    bars["open_time"] = pd.to_datetime(bars["open_time"], unit="ms", utc=True)
    bars["close_time"] = pd.to_datetime(bars["close_time"], unit="ms", utc=True)
    numeric = ["open", "high", "low", "close", "volume", "quote_volume", "trade_count",
               "taker_buy_volume", "taker_buy_quote_volume"]
    bars[numeric] = bars[numeric].apply(pd.to_numeric, errors="raise")
    return bars.set_index("open_time").sort_index()

expected_index = pd.date_range(START, END - pd.Timedelta(minutes=5), freq="5min")
if DATA_SOURCE == "binance":
    bars = load_binance_5m()
    true_jumps = None
else:
    bars, true_jumps = sb.synthetic_5m(expected_index, trends=True, jumps=True)
    print("SYNTHETIC data: results below say nothing about BTC.")

assert bars.index.is_unique
assert bars.index.equals(expected_index), "Unexpected candle coverage"
assert (bars[["open", "high", "low", "close"]] > 0).all().all()
print("Rows:", len(bars), bars.index.min(), "->", bars.index.max())

Loaded 5,000 candles

Loaded 105,984 candles
Rows: 105984 2024-12-30 00:00:00+00:00 -> 2026-01-01 23:55:00+00:00


## Freeze the reference price movements

We compute **one** E0.3c price-impulse reference table per timeframe, from prices and ATR—not bands. All (N,σ) variations use that same table. `onset_index` and `confirmation_index` represent retrospective reference boundaries.

In [ ]:
def as_utc(value):
    ts = pd.Timestamp(value)
    if ts.tzinfo is None:
        return ts.tz_localize("UTC")
    return ts.tz_convert("UTC")


runs = {}

for label, step in TF.items():
    early = as_utc(START) - pd.Timedelta("15D")

    data = bollinger_from_5m(
        bars,
        early,
        as_utc(END),
        resolution=step,
        window=2,
        num_std=2.0,
    )

    refs = price_impulse_references(
        data,
        lookback=3,
        threshold_atr=1.5,
        cooldown_bars=6,
    )

    mask = (
        (refs["onset"] >= as_utc(SCORE_START))
        & (refs["detected_at"] < as_utc(END))
    )

    refs = refs.loc[mask].copy()

    runs[label] = {
        "data": data,
        "references": refs,
    }

    print(
        label,
        "bars:", len(data),
        "fixed reference impulses:", len(refs),
    )

ValueError: Cannot pass a datetime or Timestamp with tzinfo with the tz parameter. Use tz_convert instead.

## Compare parameter surfaces

`response_fraction` is the fraction of reference impulses where the directional band moved at least **0.25 pre-onset ATR** by the reference-confirmation candle. `median_band_response_lag` is bars from reference onset to threshold, *conditional on responding*. `outside_at_confirmation` is the fraction with price still outside the contemporaneous band at reference confirmation. `recontain_fraction` considers only that outside subset, and checks the following 6 bars.

**Caution:** Instant band recontainment can suppress subsequent crossing alarms. Band tracking is therefore a diagnostic property, not automatically a desirable one.

In [ ]:
surfaces, details = {}, {}
for label, run in runs.items():
    detail = impulse_response_details(
        run['data'],run['references'],windows=WINDOWS,sigmas=SIGMAS,
        follow_bars=6,response_atr=.25,
    )
    details[label] = detail
    surfaces[label] = summarize_impulse_response(detail)
    print('\n',label, 'baseline (20,2):')
    display(surfaces[label].loc[(surfaces[label].window==20)&(surfaces[label].sigma==2.)])


## Heatmaps — response and recontainment

All heatmaps are per timeframe. Do **not** interpret a shorter conditional lag without also inspecting `response_fraction` and its reference count.

In [ ]:
METRICS=['response_fraction','median_band_response_lag',
         'median_band_move_atr','median_tracking_gap_atr',
         'outside_at_confirmation','recontain_fraction']
for label in ['1H','15m','4H','5m']:
    fig,axs=plt.subplots(2,3,figsize=(17,9),layout='constrained')
    for ax,metric in zip(axs.flat,METRICS):
        plot_heatmap(surfaces[label],metric,title=f'{label} — {metric}',ax=ax)
    plt.show()


## Inspect time-to-recontainment and its apparent mechanism

The `PRICE_DOMINANT` / `BAND_DOMINANT` split is a heuristic decomposition of movements after confirmation. Both can act together. Episodes that do not start outside the band are not counted as recontainments.

In [ ]:
for label in ['1H','15m']:
    d=details[label]
    base=d.loc[(d.window==20)&(d.sigma==2.)]
    print(label,'(20,2) recontainment outcomes')
    display(base.recontain_cause.value_counts(dropna=False).to_frame('count'))
    display(base[['direction','band_response_lag_bars','band_move_atr',
                 'price_move_atr','directional_tracking_gap_atr',
                 'outside_at_confirm','recontain_lag_bars','recontain_cause']].head(12))


## What to inspect next

1. Does the apparent 6–10 bar response improvement persist across 1H/15m after holding `sigma` fixed?
2. Which (N,σ) regions maintain reasonable previous-band containment *and* useful outside-at-confirmation visibility?
3. Is recontainment mostly price retracement or envelope expansion?
4. Only after these checks, reuse frozen E0.3c references for Pd/false-alarm/latency sweeps. This notebook **does not** rank winners or demonstrate a trading edge.